In [ ]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import KFold, GridSearchCV, cross_val_predict
from sklearn.linear_model import ElasticNet
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.feature_selection import RFE
from sklearn.tree import DecisionTreeRegressor
from sklearn.compose import TransformedTargetRegressor
# Configuração para o pandas exibir os dataframes de forma mais amigável no Colab
pd.set_option('display.max_columns', None)
caminho_pasta = ""

In [ ]:
def criar_precessador(num_features, cat_features):
    num_transformer = StandardScaler()
    cat_transformer = OneHotEncoder(handle_unknown='ignore')

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', num_transformer, num_features),
            ('cat', cat_transformer, cat_features)
        ])
    return preprocessor

In [ ]:
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import ElasticNet
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import numpy as np

def configurar_experimentos_fase1():
    """
    Retorna o dicionário de modelos focado na comparação das
    transformações básicas: Original, Logarítmica e Raiz Quadrada.
    """
    experimentos = {}

    # ==========================================
    # BLOCO 1: Variável Alvo Original (Baseline)
    # ==========================================
    experimentos['ElasticNet (Original)'] = {
        'transformacao': 'Original',
        'model': ElasticNet(random_state=42),
        'params': {'feature_selection__n_features_to_select': [0.5, 0.8, 1.0], 'regressor__alpha': [0.1, 1.0, 10.0]}
    }
    experimentos['SVR (Original)'] = {
        'transformacao': 'Original',
        'model': SVR(),
        'params': {'feature_selection__n_features_to_select': [0.5, 0.8, 1.0], 'regressor__C': [0.1, 1.0, 10.0], 'regressor__kernel': ['linear', 'rbf']}
    }
    experimentos['RandomForest (Original)'] = {
        'transformacao': 'Original',
        'model': RandomForestRegressor(random_state=42),
        'params': {'feature_selection__n_features_to_select': [0.5, 0.8, 1.0], 'regressor__n_estimators': [50,75,100], 'regressor__max_depth':[3,5,10] }
    }
    experimentos['GradientBoosting (Original)'] = {
        'transformacao': 'Original',
        'model': GradientBoostingRegressor(random_state=42),
        'params': {'feature_selection__n_features_to_select': [0.5, 0.8, 1.0], 'regressor__n_estimators': [50,75,100], 'regressor__learning_rate': [0.01,0.05,0.1]}
    }

    # ==========================================
    # BLOCO 2: Transformação Logarítmica (Log1p)
    # ==========================================
    experimentos['ElasticNet (Log)'] = {
        'transformacao': 'Log',
        'model': TransformedTargetRegressor(regressor=ElasticNet(random_state=42), func=np.log1p, inverse_func=np.expm1),
        # Alphas muito baixos devido à compressão forte do Log
        'params': {'feature_selection__n_features_to_select': [0.5, 0.8, 1.0], 'regressor__regressor__alpha': [0.0001, 0.001, 0.01]}
    }
    experimentos['SVR (Log)'] = {
        'transformacao': 'Log',
        'model': TransformedTargetRegressor(regressor=SVR(), func=np.log1p, inverse_func=np.expm1),
        'params': {'feature_selection__n_features_to_select': [0.5, 0.8, 1.0], 'regressor__regressor__C': [0.1, 1.0, 10.0], 'regressor__regressor__kernel': ['linear', 'rbf']}
    }
    experimentos['RandomForest (Log)'] = {
        'transformacao': 'Log',
        'model': TransformedTargetRegressor(regressor=RandomForestRegressor(random_state=42), func=np.log1p, inverse_func=np.expm1),
        'params': {'feature_selection__n_features_to_select': [0.5, 0.8, 1.0], 'regressor__regressor__n_estimators': [50,75,100], 'regressor__regressor__max_depth': [3,5,10]}
    }
    experimentos['GradientBoosting (Log)'] = {
        'transformacao': 'Log',
        'model': TransformedTargetRegressor(regressor=GradientBoostingRegressor(random_state=42), func=np.log1p, inverse_func=np.expm1),
        'params': {'feature_selection__n_features_to_select': [0.5, 0.8, 1.0], 'regressor__regressor__n_estimators': [50,75,100], 'regressor__regressor__learning_rate': [0.01,0.05, 0.1]}
    }

    # ==========================================
    # BLOCO 3: Transformação de Raiz Quadrada
    # ==========================================
    experimentos['ElasticNet (Raiz)'] = {
        'transformacao': 'Raiz Quadrada',
        'model': TransformedTargetRegressor(regressor=ElasticNet(random_state=42), func=np.sqrt, inverse_func=np.square),
        # Alphas intermediários (entre Original e Log)
        'params': {'feature_selection__n_features_to_select': [0.5, 0.8, 1.0], 'regressor__regressor__alpha': [0.01, 0.1, 1.0]}
    }
    experimentos['SVR (Raiz)'] = {
        'transformacao': 'Raiz Quadrada',
        'model': TransformedTargetRegressor(regressor=SVR(), func=np.sqrt, inverse_func=np.square),
        'params': {'feature_selection__n_features_to_select': [0.5, 0.8, 1.0], 'regressor__regressor__C': [0.1, 1.0, 10.0], 'regressor__regressor__kernel': ['linear', 'rbf']}
    }
    experimentos['RandomForest (Raiz)'] = {
        'transformacao': 'Raiz Quadrada',
        'model': TransformedTargetRegressor(regressor=RandomForestRegressor(random_state=42), func=np.sqrt, inverse_func=np.square),
        'params': {'feature_selection__n_features_to_select': [0.5, 0.8, 1.0], 'regressor__regressor__n_estimators': [50,75,100], 'regressor__regressor__max_depth': [3,5,10]}
    }
    experimentos['GradientBoosting (Raiz)'] = {
        'transformacao': 'Raiz Quadrada',
        'model': TransformedTargetRegressor(regressor=GradientBoostingRegressor(random_state=42), func=np.sqrt, inverse_func=np.square),
        'params': {'feature_selection__n_features_to_select': [0.5, 0.8, 1.0], 'regressor__regressor__n_estimators': [50,75,100], 'regressor__regressor__learning_rate': [0.01,0.05, 0.1]}
    }

    return experimentos

### Função para Executar Pipeline com Validação Cruzada Aninhada

Abaixo, uma nova função `executar_pipeline_aninhado` que implementa a lógica de validação cruzada aninhada. Esta função usará um `KFold` externo para dividir os dados em folds de treinamento/teste e, dentro de cada fold de treinamento, executará um `GridSearchCV` (com seu próprio `KFold` interno) para otimização de hiperparâmetros.

In [ ]:
import pickle

def executar_pipeline_aninhado(X, y, num_features, cat_features, caminho_pasta):
    preprocessor = criar_precessador(num_features, cat_features)
    experimentos = configurar_experimentos_fase1()
    estimador_base_rfe = DecisionTreeRegressor(random_state=42)

    tipos_transformacao = ['Original', 'Log', 'Raiz Quadrada']

    outer_cv = KFold(n_splits=5, shuffle=True, random_state=42)
    inner_cv = KFold(n_splits=3, shuffle=True, random_state=42)

    nested_cv_mae_scores = {tipo: {nome: [] for nome in experimentos.keys() if experimentos[nome]['transformacao'] == tipo} for tipo in tipos_transformacao}
    all_predictions = {tipo: pd.DataFrame(index=y.index) for tipo in tipos_transformacao}
    all_y_true = pd.Series(index=y.index, name='Valor_Real')

    print("\n--- Iniciando Validação Cruzada Aninhada ---")
    for fold_idx, (train_index_outer, test_index_outer) in enumerate(outer_cv.split(X, y)):
        print(f"\n>>> Rodada Externa de CV: {fold_idx + 1}/{outer_cv.n_splits}")

        X_train_outer, X_test_outer = X.iloc[train_index_outer], X.iloc[test_index_outer]
        y_train_outer, y_test_outer = y.iloc[train_index_outer], y.iloc[test_index_outer]

        all_y_true.loc[test_index_outer] = y_test_outer

        for nome, config in experimentos.items():
            tipo = config['transformacao']
            print(f"  [{tipo}] Otimizando e Avaliando {nome} na Rodada Externa {fold_idx + 1}...")

            pipeline = Pipeline(steps=[
                ('preprocessor', preprocessor),
                ('feature_selection', RFE(estimator=estimador_base_rfe, step=1)),
                ('regressor', config['model'])
            ])

            grid_search = GridSearchCV(
                estimator=pipeline, param_grid=config['params'],
                cv=inner_cv, scoring='neg_mean_absolute_error', n_jobs=-1
            )
            grid_search.fit(X_train_outer, y_train_outer)

            best_model_this_fold = grid_search.best_estimator_
            y_pred_outer_test = best_model_this_fold.predict(X_test_outer)
            mae_outer_test = np.mean(np.abs(y_test_outer - y_pred_outer_test))

            nested_cv_mae_scores[tipo][nome].append(mae_outer_test)
            all_predictions[tipo].loc[test_index_outer, f'Pred_{nome}'] = y_pred_outer_test

    print("\n--- Validação Cruzada Aninhada Concluída ---")

    resultados_finais_nested = {}
    for tipo in tipos_transformacao:
        resumo_df_list = []
        for nome_modelo, maes in nested_cv_mae_scores[tipo].items():
            resumo_df_list.append({
                'Modelo': nome_modelo,
                'MAE Médio (Nested CV)': np.mean(maes),
                'MAE Std (Nested CV)': np.std(maes) if len(maes) > 1 else 0,
                'MAEs por Fold (Nested CV)': maes
            })
        df_resumo = pd.DataFrame(resumo_df_list).sort_values(by='MAE Médio (Nested CV)').reset_index(drop=True)

        df_residuos_final = pd.DataFrame({'Valor_Real': all_y_true.loc[y.index]})
        df_residuos_final = df_residuos_final.merge(all_predictions[tipo], left_index=True, right_index=True)

        for col in df_residuos_final.columns:
            if col.startswith('Pred_'):
                model_name = col.replace('Pred_', '')
                df_residuos_final[f'Erro_{model_name}'] = df_residuos_final['Valor_Real'] - df_residuos_final[col]

        nome_arquivo = tipo.lower().replace(' ', '_')
        caminho_csv = f"{caminho_pasta}/predicoes_vs_reais_{nome_arquivo}_nested_cv.csv"
        df_residuos_final.to_csv(caminho_csv, index=True)

        # NOVO: salva o resumo em disco para a exportação posterior
        caminho_resumo = f"{caminho_pasta}/resumo_{nome_arquivo}_nested_cv.csv"
        df_resumo.to_csv(caminho_resumo, index=False)

        resultados_finais_nested[tipo] = {
            'Resumo': df_resumo,
            'Residuos': df_residuos_final,
            'Caminho': caminho_csv,
            'CaminhoResumo': caminho_resumo
        }

    # NOVO: pickle do dict inteiro (opcional, mas conveniente)
    with open(f"{caminho_pasta}/resultados_aninhados.pkl", "wb") as f:
        pickle.dump(resultados_finais_nested, f)

    print("Validação cruzada aninhada concluída com sucesso!")
    return resultados_finais_nested

In [ ]:
def exportar_resumo_unico_latex(caminho_pasta, tipos_transformacao=None,
                                casas_decimais=4, agrupar_visual=True):
    if tipos_transformacao is None:
        tipos_transformacao = ['Original', 'Log', 'Raiz Quadrada']

    fmt = f"{{:.{casas_decimais}f}}".format

    partes = []
    for tipo in tipos_transformacao:
        nome_arquivo = tipo.lower().replace(' ', '_')
        caminho_resumo = f"{caminho_pasta}/resumo_{nome_arquivo}_nested_cv.csv"
        df = pd.read_csv(caminho_resumo)

        df = df.drop(columns=['MAEs por Fold (Nested CV)'], errors='ignore')
        df.insert(0, 'Transformação', tipo)  # coluna indicando a transformação
        partes.append(df)

    df_unico = pd.concat(partes, ignore_index=True)

    # Nomes de coluna mais amigáveis
    df_unico = df_unico.rename(columns={
        'MAE Médio (Nested CV)': 'MAE Médio',
        'MAE Std (Nested CV)': 'MAE (Desvio Padrão)'
    })

    # Opcional: mostra o rótulo da transformação só na primeira linha de cada grupo
    if agrupar_visual:
        df_unico['Transformação'] = df_unico['Transformação'].mask(
            df_unico['Transformação'].duplicated(), ''
        )

    latex_str = df_unico.to_latex(
        index=False,
        escape=True,
        float_format=fmt,
        column_format='ll' + 'r' * (df_unico.shape[1] - 2),
        caption='Resumo do MAE (Nested CV) por transformação e modelo.',
        label='tab:resumo_geral',
        position='htbp'
    )

    caminho_tex = f"{caminho_pasta}/tabela_resumo_geral.tex"
    with open(caminho_tex, 'w', encoding='utf-8') as f:
        f.write(latex_str)

    print(f"LaTeX salvo em: {caminho_tex}\n")
    print(latex_str)  # imprime para conferir o formato
    return caminho_tex

In [ ]:
# def executar_pipeline_triplo(X, y, num_features, cat_features, caminho_pasta):
#     preprocessor = criar_precessador(num_features, cat_features)
#     experimentos = configurar_experimentos_fase1()
#     cv_strategy = KFold(n_splits=7, shuffle=True, random_state=42)
#     estimador_base_rfe = DecisionTreeRegressor(random_state=42)

#     # 1. Lista centralizadora dos tipos de transformação (facilita manutenções futuras)
#     tipos_transformacao = ['Original', 'Log', 'Raiz Quadrada']

#     # 2. Inicialização dinâmica das estruturas de armazenamento
#     resumos = {tipo: [] for tipo in tipos_transformacao}
#     df_residuos = {tipo: pd.DataFrame({'Valor_Real': y.values}) for tipo in tipos_transformacao}

#     for nome, config in experimentos.items():
#         tipo = config['transformacao']
#         print(f"[{tipo}] Otimizando {nome}...")

#         pipeline = Pipeline(steps=[
#             ('preprocessor', preprocessor),
#             ('feature_selection', RFE(estimator=estimador_base_rfe, step=1)),
#             ('regressor', config['model'])
#         ])

#         grid_search = GridSearchCV(
#             estimator=pipeline, param_grid=config['params'],
#             cv=cv_strategy, scoring='neg_mean_absolute_error', n_jobs=-1
#         )
#         grid_search.fit(X, y)
#         melhor_modelo = grid_search.best_estimator_

#         # O cross_val_predict com TransformedTargetRegressor já retorna o valor em milímetros
#         y_pred = cross_val_predict(melhor_modelo, X, y, cv=cv_strategy)

#         # Armazena os resíduos no DataFrame correto
#         df_residuos[tipo][f'Pred_{nome}'] = y_pred
#         df_residuos[tipo][f'Erro_{nome}'] = y.values - y_pred

#         # Extração das Features do pipeline treinado
#         nomes_colunas = melhor_modelo.named_steps['preprocessor'].get_feature_names_out()
#         mascara = melhor_modelo.named_steps['feature_selection'].support_
#         features_selecionadas = nomes_colunas[mascara].tolist()

#         resumos[tipo].append({
#             'Modelo': nome,
#             'MAE Médio': -grid_search.best_score_,
#             'Qtd Features': len(features_selecionadas),
#             'Variáveis Importantes': str(features_selecionadas),
#             'Melhores Params': str(grid_search.best_params_)
#         })

#     # Converte os resumos em DataFrames e salva tudo
#     resultados_finais = {}
#     for tipo in tipos_transformacao:
#         df_res = pd.DataFrame(resumos[tipo]).sort_values(by='MAE Médio').reset_index(drop=True)

#         # Formata o nome do arquivo substituindo espaços por sublinhados
#         nome_arquivo = tipo.lower().replace(' ', '_')
#         caminho_csv = f"{caminho_pasta}/predicoes_vs_reais_{nome_arquivo}.csv"
#         df_residuos[tipo].to_csv(caminho_csv, index=False)

#         resultados_finais[tipo] = {
#             'Resumo': df_res,
#             'Residuos': df_residuos[tipo],
#             'Caminho': caminho_csv
#         }

#     print("\nExecução tripla concluída com sucesso!")
#     return resultados_finais

In [ ]:
df = pd.read_csv("dataset_de_corrosao_tratado_incluindo_ct_numerico.csv")
y = df['d max (mm)']
X = df.drop(columns=['d max (mm)'])

# Remover os outliers extremos reportados no paper e duplicatas
print("Shape antes das remoções:")
print(df.shape)
outlier_indices = [75, 80, 101, 145, 146, 183, 184, 239, 246]
df = df.drop(index=outlier_indices).reset_index(drop=True)
df = df.drop_duplicates()
print("Shape após remoções:")
print(df.shape)

cols_num = X.select_dtypes(include=['number']).columns.tolist()
cols_cat = X.select_dtypes(exclude=['number']).columns.tolist()

print(f"Total de preditores: {X.shape[1]}")
print(f"Colunas Numéricas ({len(cols_num)}): {cols_num}")
print(f"Colunas Categóricas ({len(cols_cat)}): {cols_cat}")

Shape antes das remoções:
(259, 13)
Shape após remoções:
(245, 13)
Total de preditores: 12
Colunas Numéricas (11): ['t (years)', 'pH', 'pp (V)', 're (Ω·m)', 'wc (%)', 'bd (g/mL)', 'cc (ppm)', 'bc (ppm)', 'sc (ppm)', 'rp (mV)', 'ct']
Colunas Categóricas (1): ['Class']


In [ ]:
# Treino (roda uma vez)
print("Iniciando a bateria de experimentos A/B/C com validação cruzada aninhada...")
resultados_aninhados = executar_pipeline_aninhado(X, y, cols_num, cols_cat, ".")

# Exportação (pode rodar quantas vezes quiser, sem retreinar)
caminho = exportar_resumo_unico_latex(".")


Iniciando a bateria de experimentos A/B/C com validação cruzada aninhada...

--- Iniciando Validação Cruzada Aninhada ---

>>> Rodada Externa de CV: 1/5
  [Original] Otimizando e Avaliando ElasticNet (Original) na Rodada Externa 1...
  [Original] Otimizando e Avaliando SVR (Original) na Rodada Externa 1...
  [Original] Otimizando e Avaliando RandomForest (Original) na Rodada Externa 1...
  [Original] Otimizando e Avaliando GradientBoosting (Original) na Rodada Externa 1...
  [Log] Otimizando e Avaliando ElasticNet (Log) na Rodada Externa 1...
  [Log] Otimizando e Avaliando SVR (Log) na Rodada Externa 1...
  [Log] Otimizando e Avaliando RandomForest (Log) na Rodada Externa 1...
  [Log] Otimizando e Avaliando GradientBoosting (Log) na Rodada Externa 1...
  [Raiz Quadrada] Otimizando e Avaliando ElasticNet (Raiz) na Rodada Externa 1...
  [Raiz Quadrada] Otimizando e Avaliando SVR (Raiz) na Rodada Externa 1...
  [Raiz Quadrada] Otimizando e Avaliando RandomForest (Raiz) na Rodada Externa 1

In [ ]:
import re, pickle
import numpy as np, pandas as pd
from scipy.stats import wilcoxon, binomtest

with open("./resultados_aninhados.pkl", "rb") as f:
    R = pickle.load(f)

def maes_por_fold(tipo):
    df = R[tipo]['Resumo']
    return {re.sub(r'\s*\(.*\)', '', row['Modelo']):
            np.asarray(row['MAEs por Fold (Nested CV)'], dtype=float)
            for _, row in df.iterrows()}

def comparar(base='Original', alvo='Log'):
    mb, ma = maes_por_fold(base), maes_por_fold(alvo)
    linhas = []
    for alg in mb:
        d = mb[alg] - ma[alg]                  # positivo = alvo melhor
        try:
            _, p = wilcoxon(mb[alg], ma[alg], alternative='greater')
        except ValueError:
            p = np.nan
        linhas.append({'Algoritmo': alg,
                       'Folds c/ melhora': f"{int((d > 0).sum())}/{len(d)}",
                       'Δ MAE médio': d.mean(),
                       'Wilcoxon p (unilat.)': p})
    df = pd.DataFrame(linhas)
    k = (df['Δ MAE médio'] > 0).sum()
    sinais_p = binomtest(int(k), len(df), 0.5, alternative='greater').pvalue
    return df, sinais_p

for alvo in ['Log', 'Raiz Quadrada']:
    tab, p_sinais = comparar('Original', alvo)
    print(f"\n=== Original vs {alvo} ===")
    display(tab.round(4))
    print(f"Teste de sinais entre algoritmos: {int((tab['Δ MAE médio']>0).sum())}/{len(tab)}, p = {p_sinais:.4f}")


=== Original vs Log ===


,Algoritmo,Folds c/ melhora,Δ MAE médio,Wilcoxon p (unilat.)
0,SVR,2/5,0.0184,0.5000
1,RandomForest,5/5,0.1104,0.0312
2,GradientBoosting,5/5,0.1363,0.0312
3,ElasticNet,5/5,0.0914,0.0312


Teste de sinais entre algoritmos: 4/4, p = 0.0625

=== Original vs Raiz Quadrada ===


,Algoritmo,Folds c/ melhora,Δ MAE médio,Wilcoxon p (unilat.)
0,SVR,2/5,0.0239,0.5000
1,RandomForest,4/5,0.0823,0.0938
2,GradientBoosting,4/5,0.1054,0.0625
3,ElasticNet,5/5,0.0889,0.0312


Teste de sinais entre algoritmos: 4/4, p = 0.0625


### Executando a Validação Cruzada Aninhada

Agora vamos executar a nova função `executar_pipeline_aninhado` e exibir os resumos de desempenho para cada tipo de transformação.

In [ ]:
print("Iniciando a bateria de experimentos A/B/C com validação cruzada aninhada...")
resultados_aninhados = executar_pipeline_aninhado(X, y, cols_num, cols_cat, "")
display(resultados_aninhados['Original']['Resumo'])
display(resultados_aninhados['Log']['Resumo'])
display(resultados_aninhados['Raiz Quadrada']['Resumo'])

Iniciando a bateria de experimentos A/B/C com validação cruzada aninhada...

--- Iniciando Validação Cruzada Aninhada ---

>>> Rodada Externa de CV: 1/5
  [Original] Otimizando e Avaliando ElasticNet (Original) na Rodada Externa 1...
  [Original] Otimizando e Avaliando SVR (Original) na Rodada Externa 1...
  [Original] Otimizando e Avaliando RandomForest (Original) na Rodada Externa 1...
  [Original] Otimizando e Avaliando GradientBoosting (Original) na Rodada Externa 1...
  [Log] Otimizando e Avaliando ElasticNet (Log) na Rodada Externa 1...
  [Log] Otimizando e Avaliando SVR (Log) na Rodada Externa 1...
  [Log] Otimizando e Avaliando RandomForest (Log) na Rodada Externa 1...
  [Log] Otimizando e Avaliando GradientBoosting (Log) na Rodada Externa 1...
  [Raiz Quadrada] Otimizando e Avaliando ElasticNet (Raiz) na Rodada Externa 1...
  [Raiz Quadrada] Otimizando e Avaliando SVR (Raiz) na Rodada Externa 1...
  [Raiz Quadrada] Otimizando e Avaliando RandomForest (Raiz) na Rodada Externa 1

,Modelo,MAE Médio (Nested CV),MAE Std (Nested CV),MAEs por Fold (Nested CV)
0,SVR (Original),0.809891,0.188263,"[0.6851020755182969, 0.7816404619580702, 1.041..."
1,GradientBoosting (Original),0.906139,0.104379,"[1.0722150949920937, 0.8193004013543373, 0.851..."
2,RandomForest (Original),0.915199,0.119183,"[0.9770559214313729, 0.836661007747084, 0.8593..."
3,ElasticNet (Original),1.026763,0.060523,"[0.9584394378118744, 1.0113404671766208, 1.045..."


,Modelo,MAE Médio (Nested CV),MAE Std (Nested CV),MAEs por Fold (Nested CV)
0,SVR (Log),0.791486,0.110316,"[0.7276439490048614, 0.7956630126569617, 0.867..."
1,GradientBoosting (Log),0.798325,0.095121,"[0.7970893426112364, 0.8570847872664168, 0.856..."
2,RandomForest (Log),0.806667,0.111705,"[0.7030833584038907, 0.8518980615534775, 0.809..."
3,ElasticNet (Log),0.935404,0.083517,"[0.7763986918177138, 0.9419931385600484, 1.020..."


,Modelo,MAE Médio (Nested CV),MAE Std (Nested CV),MAEs por Fold (Nested CV)
0,SVR (Raiz),0.785996,0.112525,"[0.7269497069645381, 0.7936552358885377, 0.855..."
1,RandomForest (Raiz),0.823856,0.117003,"[0.74890985292621, 0.8398771666298848, 0.83317..."
2,GradientBoosting (Raiz),0.829382,0.088556,"[0.8329341385018983, 0.8639737741036307, 0.835..."
3,ElasticNet (Raiz),0.937896,0.087591,"[0.7711852537733931, 0.9507899816990087, 1.021..."


In [ ]:
def analisar_mae_por_extrato_dinamico(df_residuos, faixas, coluna_alvo='Valor_Real'):
    colunas_erro = [c for c in df_residuos.columns if c.startswith('Erro_')]
    modelos = [c.replace('Erro_', '') for c in colunas_erro]
    print(f"Modelos detectados automaticamente: {modelos}\n")

    df = df_residuos.copy()
    df['Faixa'] = pd.cut(df[coluna_alvo], bins=faixas)

    linhas = []
    for faixa, g in df.groupby('Faixa', observed=False):
        media_alvo = g[coluna_alvo].mean()
        linha = {'Faixa': str(faixa), 'Amostras': len(g)}
        for m in modelos:
            mae = g[f'Erro_{m}'].abs().mean()
            linha[f'MAE_{m}'] = mae
            linha[f'MAErel_{m}'] = (mae / media_alvo * 100) if media_alvo else np.nan
        linhas.append(linha)
    return pd.DataFrame(linhas)


In [ ]:
def exportar_extratos_latex(df_analise, caminho_tex, modelos=None,
                            incluir_relativo=True, casas=3, casas_rel=1):
    df = df_analise.copy()
    if modelos is None:
        modelos = [c.replace('MAE_', '') for c in df.columns if c.startswith('MAE_')]

    disp = pd.DataFrame({'Faixa': df['Faixa'], 'Amostras': df['Amostras']})
    for m in modelos:
        disp[f'MAE {m}'] = df[f'MAE_{m}'].map(
            lambda x: '--' if pd.isna(x) else f'{x:.{casas}f}')
        if incluir_relativo:
            disp[f'MAE rel. {m} (\\%)'] = df[f'MAErel_{m}'].map(
                lambda x: '--' if pd.isna(x) else f'{x:.{casas_rel}f}')

    latex = disp.to_latex(
        index=False, escape=False,   # nomes já formatados; \% e -- controlados manualmente
        caption='MAE absoluto e relativo por faixa de dmax (previsões out-of-fold).',
        label='tab:extratos_dmax', position='htbp')

    with open(caminho_tex, 'w', encoding='utf-8') as f:
        f.write(latex)
    print(f"LaTeX salvo em: {caminho_tex}\n")
    print(latex)
    return caminho_tex

In [ ]:
faixas =  [0, 2, 4, 6, 15]   # ajuste aos seus dmax
df_res = pd.read_csv("./predicoes_vs_reais_log_nested_cv.csv", index_col=0)

analise = analisar_mae_por_extrato_dinamico(df_res, faixas)
display(analise)

exportar_extratos_latex(analise, "./tabela_extratos_dmax.tex",
                        modelos=['GradientBoosting (Log)'])

Modelos detectados automaticamente: ['ElasticNet (Log)', 'SVR (Log)', 'RandomForest (Log)', 'GradientBoosting (Log)']



,Faixa,Amostras,MAE_ElasticNet (Log),MAErel_ElasticNet (Log),MAE_SVR (Log),MAErel_SVR (Log),MAE_RandomForest (Log),MAErel_RandomForest (Log),MAE_GradientBoosting (Log),MAErel_GradientBoosting (Log)
0,"(0, 2]",186,0.523688,48.547625,0.463840,42.999481,0.481640,44.649627,0.508271,47.118417
1,"(2, 4]",43,0.729846,26.593845,0.832981,30.351823,0.856547,31.210495,0.842942,30.714783
2,"(4, 6]",14,1.906091,38.736071,1.344966,27.332742,1.371718,27.876401,1.320891,26.843483
3,"(6, 15]",16,5.422230,63.431805,4.015039,46.969822,3.937207,46.059305,3.348686,39.174505


LaTeX salvo em: ./tabela_extratos_dmax.tex

\begin{table}[htbp]
\caption{MAE absoluto e relativo por faixa de dmax (previsões out-of-fold).}
\label{tab:extratos_dmax}
\begin{tabular}{lrll}
\toprule
Faixa & Amostras & MAE GradientBoosting (Log) & MAE rel. GradientBoosting (Log) (\%) \\
\midrule
(0, 2] & 186 & 0.508 & 47.1 \\
(2, 4] & 43 & 0.843 & 30.7 \\
(4, 6] & 14 & 1.321 & 26.8 \\
(6, 15] & 16 & 3.349 & 39.2 \\
\bottomrule
\end{tabular}
\end{table}



'./tabela_extratos_dmax.tex'

In [ ]:
from functools import reduce
import pandas as pd

# 1. Atualizando as fronteiras (bins) conforme sua nova definição
bins_estratificacao = [0, 2, 4, 6, 15]

# 2. Gerando os extratos dinamicamente para cada experimento
lista_dfs_extratos = []

# Agora usamos o dicionário resultados_aninhados
for tipo in resultados_aninhados.keys():
    print(f"Processando extratos para: {tipo} (Validação Aninhada)...")
    df_extrato = analisar_mae_por_extrato_dinamico(resultados_aninhados[tipo]['Residuos'], bins_estratificacao)
    lista_dfs_extratos.append(df_extrato)

# 3. Mesclando múltiplos DataFrames de forma elegante
df_comparativo_aninhado = reduce(
    lambda esq, dir: pd.merge(esq, dir, on=['Faixa', 'Amostras']),
    lista_dfs_extratos
)

print("\n" + "=" * 80)
print("🚀 COMPARAÇÃO ESTRATIFICADA VALIDAÇÃO CRUZADA ANINHADA: ORIGINAL vs LOG vs RAIZ (MAE em mm)")
print("=" * 80)
display(df_comparativo_aninhado)

# Opcional: Salvar no Drive para o grupo analisar no Excel/Planilhas
caminho_comparativo_aninhado = f"/comparativo_estratificado_aninhado_cv.csv"
df_comparativo_aninhado.to_csv(caminho_comparativo_aninhado, index=False)
print(f"\nArquivo comparativo salvo em: {caminho_comparativo_aninhado}")

Processando extratos para: Original (Validação Aninhada)...
Modelos detectados automaticamente: ['ElasticNet (Original)', 'SVR (Original)', 'RandomForest (Original)', 'GradientBoosting (Original)']

Processando extratos para: Log (Validação Aninhada)...
Modelos detectados automaticamente: ['ElasticNet (Log)', 'SVR (Log)', 'RandomForest (Log)', 'GradientBoosting (Log)']

Processando extratos para: Raiz Quadrada (Validação Aninhada)...
Modelos detectados automaticamente: ['ElasticNet (Raiz)', 'SVR (Raiz)', 'RandomForest (Raiz)', 'GradientBoosting (Raiz)']


🚀 COMPARAÇÃO ESTRATIFICADA VALIDAÇÃO CRUZADA ANINHADA: ORIGINAL vs LOG vs RAIZ (MAE em mm)


,Faixa,Amostras,MAE_ElasticNet (Original),MAE_SVR (Original),MAE_RandomForest (Original),MAE_GradientBoosting (Original),MAE_ElasticNet (Log),MAE_SVR (Log),MAE_RandomForest (Log),MAE_GradientBoosting (Log),MAE_ElasticNet (Raiz),MAE_SVR (Raiz),MAE_RandomForest (Raiz),MAE_GradientBoosting (Raiz)
0,"(0, 2]",186,0.749378,0.530727,0.694597,0.694739,0.523688,0.463840,0.491299,0.508383,0.572948,0.479625,0.541573,0.565103
1,"(2, 4]",43,0.599236,0.789623,0.932562,0.874310,0.729846,0.832981,0.830754,0.835089,0.693197,0.803605,0.863665,0.780037
2,"(4, 6]",14,1.589304,1.178701,1.079299,1.288074,1.906091,1.344966,1.280953,1.470404,1.747053,1.263661,1.130830,1.470569
3,"(6, 15]",16,4.910722,3.803646,3.297563,3.121386,5.422230,4.015039,4.001026,3.493515,5.129187,3.893181,3.739198,3.483068



Arquivo comparativo salvo em: /comparativo_estratificado_aninhado_cv.csv


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Definimos a hierarquia para organizar o menu
base_modelos = ['ElasticNet', 'SVR', 'RandomForest', 'GradientBoosting']
tipos_transformacao = ['Original', 'Log', 'Raiz Quadrada']

# 2. Cria a base da figura
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=("Atual vs. Predito", "Distribuição dos Erros (Resíduos)"))

combinacoes = []

# Referência global para a Linha Ideal (y=x)
# Usamos resultados_aninhados agora
df_referencia = resultados_aninhados['Original']['Residuos']
min_val = df_referencia['Valor_Real'].min() * 0.9
max_val = df_referencia['Valor_Real'].max() * 1.1

# 3. Adiciona TODOS os 24 gráficos invisíveis na figura
for modelo in base_modelos:
    for tipo in tipos_transformacao:

        # === CORREÇÃO DO NOME (O 'if' que resolve o KeyError) ===
        if tipo == 'Raiz Quadrada':
            nome_completo = f"{modelo} (Raiz)"
        else:
            nome_completo = f"{modelo} ({tipo})"

        combinacoes.append(nome_completo)

        # Pega o DataFrame correto daquela transformação (agora de resultados_aninhados)
        df = resultados_aninhados[tipo]['Residuos']

        # Gráfico A: Dispersão
        fig.add_trace(
            go.Scatter(x=df['Valor_Real'],
                       y=df[f'Pred_{nome_completo}'],
                       mode='markers',
                       name=f'{nome_completo} - Predição',
                       marker=dict(size=8, opacity=0.7, line=dict(width=1, color='DarkSlateGrey')),
                       hovertemplate='<b>Real:</b> %{x:.2f}<br><b>Predito:</b> %{y:.2f}<br><b>Erro:</b> %{customdata:.2f}<extra></extra>',
                       customdata=df[f'Erro_{nome_completo}'],
                       visible=False),
            row=1, col=1
        )

        # Gráfico B: Histograma
        fig.add_trace(
            go.Histogram(x=df[f'Erro_{nome_completo}'],
                         name=f'{nome_completo} - Erro',
                         nbinsx=20,
                         marker_color='#1f77b4',
                         opacity=0.8,
                         visible=False),
            row=1, col=2
        )
# 4. Adiciona a "Linha Ideal (y=x)" por último
fig.add_trace(
    go.Scatter(x=[min_val, max_val], y=[min_val, max_val],
               mode='lines',
               name='Linha de Identidade (y=x)',
               line=dict(color='red', dash='dash'),
               hoverinfo='skip',
               visible=True), # Sempre visível
    row=1, col=1
)

# Liga apenas os dois primeiros gráficos (ElasticNet Original) para a visualização inicial
fig.data[0].visible = True
fig.data[1].visible = True

# 5. Lógica do Botão Dropdown Interativo
botoes = []
total_traces_dinamicos = len(combinacoes) * 2

for i, nome in enumerate(combinacoes):
    # Cria uma lista de False para todos os gráficos dinâmicos, e True para a linha estática no final
    visiveis = [False] * total_traces_dinamicos + [True]

    # Liga apenas o Scatter e o Histograma correspondentes à iteração atual
    visiveis[i * 2] = True
    visiveis[i * 2 + 1] = True

    botao = dict(
        label=nome,
        method='update',
        args=[{'visible': visiveis},
              {'title': f'Análise de Resíduos: {nome}'}]
    )
    botoes.append(botao)

# 6. Formatação do Layout
fig.update_layout(
    updatemenus=[dict(
        active=0,
        buttons=botoes,
        x=0.0, xanchor='left',
        y=1.2, yanchor='top',
        bgcolor='white', bordercolor='gray'
    )],
    title_text=f"Análise de Resíduos (Validação Aninhada): {combinacoes[0]}",
    height=550,
    showlegend=False,
    plot_bgcolor='white',
    hovermode='closest'
)

# Ajuste dos Eixos
fig.update_xaxes(title_text="Valor Real de d max (mm)", showgrid=True, gridcolor='lightgray', row=1, col=1)
fig.update_yaxes(title_text="Valor Predito", showgrid=True, gridcolor='lightgray', row=1, col=1)
fig.update_xaxes(title_text="Erro (Real - Predito)", showgrid=True, gridcolor='lightgray', row=1, col=2)
fig.update_yaxes(title_text="Frequência (Nº de Amostras)", showgrid=True, gridcolor='lightgray', row=1, col=2)

# Exibe no Notebook
fig.show()

# ==========================================
# Exportação para o Grupo
# ==========================================
# Usando o PATH_PROJETO para garantir que salve no Drive compartilhado
caminho_html_aninhado = f"/dashboard_residuos_aninhado_cv.html"
fig.write_html(caminho_html_aninhado)
print(f"Painel interativo exportado com sucesso para: {caminho_html_aninhado}")

Painel interativo exportado com sucesso para: /dashboard_residuos_aninhado_cv.html
